In [1]:
import pandas as pd
from typing import Dict, List, Tuple
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from OprFuncs import *
#from langchain.schema.runnable import RunnableSequence
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#from langchain.agents import AgentExecutor, Tool, create_react_agent
#from langchain import hub
import re
#from modelEXT.PygalCodeComponents import PygalCodeComponents
#from langchain.output_parsers import PydanticOutputParser
from DatabaseManager import DatabaseManager
from langchain_experimental.agents import create_pandas_dataframe_agent

class DataAnalyzer:
    def __init__(self,dataframe,llm,user_id=None):
        self.dataframe = dataframe
        self.llm = llm
        self.data_info = data_infer(dataframe)
        self.data_description = data_describer(dataframe)
        self.data_sample = dataframe.head().to_string()
        self.data_cols = ", ".join(dataframe.columns)
        self.db = DatabaseManager()
        self.report_id = None
        self.memory = []
        
        if user_id:
            self.user_id = user_id
            self.user_context = self.db.get_user_context(user_id)
            if self.user_context:
                self.memory.append(HumanMessage(content=f"User Context: {self.user_context}"))
        else:
            self.user_context = None

    def analysis_data(self):
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description

        analysis_template = '''
        You are a data analyst. You are provided with:
        1. Dataset metadata: {data_info}
        2. Dataset sample: {data_sample}
        3. Dataset summary: {data_description}
        4. User_context: {user_context}

        You are a highly skilled professional data analyst specialized in business data analysis.

        Given the following dataset analysis, your tasks are:
        1. Provide a **deep, comprehensive analysis** of the data.
        2. **Explain key findings**, trends, patterns, and anomalies in a meaningful way.
        3. **Interpret** what the numbers and statistics mean for the business context (not just describe them).
        4. **Identify**:
        - Critical KPIs (Key Performance Indicators).
        - Potential risks and problems suggested by the data.
        - Opportunities for growth, improvement, or efficiency.
        5. Highlight **hidden insights** that may not be immediately obvious.
        6. Make sure your analysis tells a **clear, logical story** about the business situation.

        Instructions:
        - Be detailed but concise.
        - Avoid listing plain statistics — always explain their implications.
        - Connect different findings where relevant to create a full picture.
        - Think like a business consultant, not just a data scientist.
        '''
        analysis_prompt = PromptTemplate(
            input_variables=["data_info", "data_sample", "data_description", "user_context"],
            template=analysis_template
        )
        
        analysis_chain = analysis_prompt | self.llm

        self.analysis = analysis_chain.invoke({
            "data_info": data_info,
            "data_sample": data_sample,
            "data_description": data_description,
            "user_context":self.user_context or "No prior context available"
        })

        formatted_analysis_prompt = analysis_template.format(data_info=data_info,data_sample=data_sample,
                                                            data_description=data_description,
                                                            user_context=self.user_context)
        self.memory.append(HumanMessage(content=formatted_analysis_prompt))
        self.memory.append(AIMessage(content=self.analysis))
        self.db.saveMemory(reportID=self.report_id,
                        llm=self.db.llm_id_by_name(self.llm.model),
                        prompet=formatted_analysis_prompt,
                        response=self.analysis,
                        chat=False)
        self.generate_user_context()
        return self.analysis

In [6]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
analyzer = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
analysis_result = analyzer.analysis_data()

# Print the analysis
print(analysis_result)


As a seasoned data analyst, I will delve into the provided dataset and uncover valuable insights that can inform business decisions. Here's my comprehensive analysis:

**Data Overview**
The dataset contains 852 entries with 20 columns, primarily related to soccer matches. The data spans from 1930 to 2014, providing a unique opportunity to analyze trends over an extended period.

**Key Findings**

1. **Trends in Home Team Goals**: The mean home team goals per match is 1.81, while the median is 2. This suggests that most matches see at least two goals scored by the home team. There is no significant change in this metric over time.
2. **Away Team Goals**: The mean away team goals per match is 1.02, with a median of 1. This indicates that away teams tend to score fewer goals than home teams. The standard deviation (1.09) suggests some variation, but overall, away teams struggle to score.
3. **Attendance**: The average attendance per match is approximately 45,165, with a significant spread